# 05 ML GNN Embeddings

This notebook uses `MLTrainAndStore` to train regressors with only the GraphSAGE embedding features.

In [12]:
from pathlib import Path
import sys
import os

sys.path.insert(0, os.path.abspath('../..'))

import pandas as pd

from sklearn.ensemble import ExtraTreesRegressor, HistGradientBoostingRegressor, RandomForestRegressor
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.neural_network import MLPRegressor
from sklearn.svm import SVR
import xgboost as xgb
from xgboost import XGBRegressor

from src.models.ml_train_and_store import (
    MLTrainAndStore,
    load_gnn_ml_dataset,
    make_log_regression_model,
)

pd.set_option("display.max_columns", 200)
PROJECT_ROOT = Path().resolve().parents[1]

In [13]:
print(f"Project root: {PROJECT_ROOT}")

Project root: C:\Users\ruben\Desktop\Universidade\Nova IMS\Tese\Thesis


## Load Dataset

In [14]:
df, feature_cols = load_gnn_ml_dataset(PROJECT_ROOT,filename="graphsage_srisk_dataset.parquet")
print(df.shape, len(feature_cols))

df_1, feature_cols_1 = load_gnn_ml_dataset(PROJECT_ROOT, target_col="log_systemic_risk_label", filename="node2vec_srisk_dataset.parquet")
print(df_1.shape, len(feature_cols_1))

(145536, 69) 64
(145536, 69) 64


In [15]:
trainer = MLTrainAndStore(
    df=df,
    feature_cols=feature_cols,
    target_col="log_systemic_risk_label",
)

trainer.train_df.shape, trainer.val_df.shape, trainer.test_df.shape

((109152, 69), (18192, 69), (13644, 69))

In [16]:
trainer_1 = MLTrainAndStore(
    df=df_1,
    feature_cols=feature_cols_1,
    target_col="log_systemic_risk_label",
)

trainer_1.train_df.shape, trainer_1.val_df.shape, trainer_1.test_df.shape

((109152, 69), (18192, 69), (13644, 69))

## Define Models

In [17]:
candidate_models = {
    "linear_regression": make_log_regression_model(LinearRegression(), scale_features=True),
}

list(candidate_models)

['linear_regression']

## Train And Store

In [18]:
trainer.train_many(candidate_models)

,model,train_mae,validation_mae,test_mae,train_rmse,validation_rmse,test_rmse,train_r2,validation_r2,test_r2
0,linear_regression,0.040154,0.070039,0.077018,0.142469,0.257846,0.248596,0.349214,-0.191377,-0.529112


In [19]:
trainer_1.train_many(candidate_models)

,model,train_mae,validation_mae,test_mae,train_rmse,validation_rmse,test_rmse,train_r2,validation_r2,test_r2
0,linear_regression,0.046425,0.058917,0.053222,0.162794,0.24375,0.213287,0.150283,-0.064679,-0.125594


In [20]:
trainer.results()

,model,train_mae,validation_mae,test_mae,train_rmse,validation_rmse,test_rmse,train_r2,validation_r2,test_r2
0,linear_regression,0.040154,0.070039,0.077018,0.142469,0.257846,0.248596,0.349214,-0.191377,-0.529112


In [21]:

trainer_1.results()

,model,train_mae,validation_mae,test_mae,train_rmse,validation_rmse,test_rmse,train_r2,validation_r2,test_r2
0,linear_regression,0.046425,0.058917,0.053222,0.162794,0.24375,0.213287,0.150283,-0.064679,-0.125594


## Single-Model Pattern

In [22]:
amodel = make_log_regression_model(
    XGBRegressor(n_estimators=100, learning_rate=0.1, max_depth=6, random_state=42),
    scale_features=True,
)

trainer.train_and_store(model=amodel, name="XGBRegressor_search_1")
trainer.results()


,model,train_mae,validation_mae,test_mae,train_rmse,validation_rmse,test_rmse,train_r2,validation_r2,test_r2
0,XGBRegressor_search_1,0.01069,0.035061,0.028222,0.059711,0.234415,0.194974,0.885685,0.015307,0.059403
1,linear_regression,0.040154,0.070039,0.077018,0.142469,0.257846,0.248596,0.349214,-0.191377,-0.529112


In [23]:

trainer_1.train_and_store(model=amodel, name="XGBRegressor_search_1")
trainer_1.results()

,model,train_mae,validation_mae,test_mae,train_rmse,validation_rmse,test_rmse,train_r2,validation_r2,test_r2
0,XGBRegressor_search_1,0.013105,0.035861,0.028545,0.062352,0.201231,0.157022,0.875348,0.27436,0.389941
1,linear_regression,0.046425,0.058917,0.053222,0.162794,0.24375,0.213287,0.150283,-0.064679,-0.125594


## Best Model

In [24]:
trainer.best_model_name()

'XGBRegressor_search_1'

In [25]:

trainer_1.best_model_name()

'XGBRegressor_search_1'

In [26]:
trainer.predict_test().head(20)

,bank_id,year,quarter,period,log_systemic_risk_label,prediction,abs_error
0,8,2023,1,2023Q1,3.988984,0.973939,3.015045
1,4161,2023,3,2023Q3,0.693147,3.650921,2.957774
2,4012,2023,2,2023Q2,0.693147,3.581093,2.887946
3,5,2023,1,2023Q1,4.290459,1.691636,2.598823
4,0,2023,1,2023Q1,4.043051,1.494107,2.548944
5,6,2023,1,2023Q1,3.761200,1.245316,2.515884
6,17,2023,1,2023Q1,3.806662,1.291163,2.515500
7,4,2023,1,2023Q1,3.688879,1.187836,2.501043
8,1502,2023,1,2023Q1,0.693147,3.147248,2.454101
9,531,2023,2,2023Q2,0.693147,3.123291,2.430143


In [27]:
trainer_1.predict_test().head(20)

,bank_id,year,quarter,period,log_systemic_risk_label,prediction,abs_error
0,1,2023,1,2023Q1,3.737670,1.066635,2.671035
1,4,2023,1,2023Q1,3.688879,1.062623,2.626256
2,2,2023,1,2023Q1,3.713572,1.141999,2.571573
3,0,2023,1,2023Q1,4.043051,1.476404,2.566647
4,5,2023,1,2023Q1,4.290459,1.846756,2.443704
5,7,2023,1,2023Q1,3.610918,1.212568,2.398350
6,6,2023,1,2023Q1,3.761200,1.383424,2.377777
7,2,2023,2,2023Q2,3.465736,1.205097,2.260638
8,17,2023,1,2023Q1,3.806662,1.559243,2.247419
9,8,2023,1,2023Q1,3.988984,1.750570,2.238414
